In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel, RobertaModel
from dataclasses import dataclass
from typing import Optional, Tuple, Any, Dict, List
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda'

In [ ]:
MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
STRIDE = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

evasion_classes = [
    'Declining to answer', 'Dodging', 'General', 'Explicit', 
    'Claims ignorance', 'Clarification', 'Implicit', 
    'Partial/half-answer', 'Deflection'
]
evasion_label2id = {label: idx for idx, label in enumerate(evasion_classes)}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}

In [ ]:
# Load datasets
ds = load_dataset("ailsntua/QEvasion")
train_df = ds['train'].to_pandas()
fake_test_df = ds['test'].to_pandas()

print("Train columns:", train_df.columns.tolist())
print("Fake test columns:", fake_test_df.columns.tolist())
print(f"\nTrain size: {len(train_df)}")
print(f"Fake test size: {len(fake_test_df)}")

# Derive evasion_label for fake test using majority vote
def get_majority_evasion_label(row):
    """Get majority vote from the 3 annotators for evasion label"""
    votes = [row['annotator1'], row['annotator2'], row['annotator3']]
    valid_votes = [v for v in votes if v and pd.notna(v) and v != '']
    if not valid_votes:
        return None
    counter = Counter(valid_votes)
    return counter.most_common(1)[0][0]

fake_test_df['evasion_label'] = fake_test_df.apply(get_majority_evasion_label, axis=1)

print(f"\nClarity label distribution in train:")
print(train_df['clarity_label'].value_counts())
print(f"\nEvasion label distribution in train:")
print(train_df['evasion_label'].value_counts())

print(f"\nClarity label distribution in fake test:")
print(fake_test_df['clarity_label'].value_counts())
print(f"\nEvasion label distribution in fake test:")
print(fake_test_df['evasion_label'].value_counts())

In [ ]:
# Create 85/15 stratified train/validation split
train_indices, val_indices = train_test_split(
    range(len(train_df)),
    test_size=0.15,
    random_state=SEED,
    stratify=train_df['clarity_label']
)

train_split_df = train_df.iloc[train_indices].reset_index(drop=True)
val_split_df = train_df.iloc[val_indices].reset_index(drop=True)

print(f"Train split: {len(train_split_df)} samples")
print(f"Val split: {len(val_split_df)} samples")
print(f"\nTrain clarity distribution:")
print(train_split_df['clarity_label'].value_counts())
print(f"\nVal clarity distribution:")
print(val_split_df['clarity_label'].value_counts())

## Tokenization with Question in Every Chunk

Strategy:
1. Tokenize question and answer separately
2. Chunk the answer with sliding window
3. Prepend question tokens to each answer chunk
4. Store as a list of token sequences per sample

In [ ]:
def tokenize_with_question_per_chunk(examples, has_labels=True):
    """
    Tokenize so that each chunk contains: Question + [chunk of Answer]
    """
    all_input_ids = []
    all_attention_masks = []
    
    for q, a in zip(examples["question"], examples["interview_answer"]):
        # Format: "Question: {q}\nAnswer: {a}"
        q_text = f"Question: {q}\nAnswer: "
        
        # Tokenize question prefix
        q_tokens = tokenizer(
            q_text,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]
        
        # Tokenize answer
        a_tokens = tokenizer(
            a,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]
        
        # Calculate space available for answer in each chunk
        # MAX_LENGTH - 1 (for <s>) - 1 (for </s>) - len(q_tokens)
        available_for_answer = MAX_LENGTH - 2 - len(q_tokens)
        
        if available_for_answer <= 0:
            # Question is too long, truncate it
            q_tokens = q_tokens[:MAX_LENGTH - 2]
            available_for_answer = 0
        
        # Create chunks of answer
        answer_chunks = []
        if len(a_tokens) <= available_for_answer:
            # Answer fits in one chunk
            answer_chunks.append(a_tokens)
        else:
            # Create sliding window chunks of answer
            start = 0
            while start < len(a_tokens):
                end = min(start + available_for_answer, len(a_tokens))
                answer_chunks.append(a_tokens[start:end])
                start += STRIDE
                if end >= len(a_tokens):
                    break
        
        # Combine question + answer chunks with special tokens
        # Format: <s> + q_tokens + answer_chunk + </s>
        chunk_input_ids = []
        for answer_chunk in answer_chunks:
            chunk = [tokenizer.bos_token_id] + q_tokens + answer_chunk + [tokenizer.eos_token_id]
            chunk_input_ids.append(chunk)
        
        # Flatten all chunks for this sample into one sequence (will be processed later)
        # We'll store them as a continuous sequence with a special separator
        # Actually, we need to store metadata about chunk boundaries
        # For simplicity, let's just concatenate and let the model handle chunking dynamically
        # OR we can store the first chunk only and recreate chunks in the model
        
        # Better approach: store the full Q+A concatenated, and chunk in the model
        # But we want Q in every chunk, so we need to store Q and A separately
        
        # Let's store the full sequence and handle chunking in the model's forward pass
        # We'll tokenize the full Q+A and pass metadata about where Q ends
        
        # Actually, let's just store the normal full sequence tokenization
        # and handle the Q-per-chunk logic in the model's create_chunks method
        full_input = f"Question: {q}\nAnswer: {a}"
        tokenized = tokenizer(
            full_input,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )
        
        all_input_ids.append(tokenized["input_ids"])
        all_attention_masks.append(tokenized["attention_mask"])
    
    result = {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
    }
    
    if has_labels:
        result["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
        result["evasion_labels"] = [evasion_label2id[l] for l in examples["evasion_label"]]
    else:
        result["clarity_labels"] = [-100] * len(examples["question"])
        result["evasion_labels"] = [-100] * len(examples["question"])
    
    # Store question boundaries for chunk reconstruction
    # Store the length of "Question: {q}\nAnswer: " tokens
    question_prefix_lengths = []
    for q in examples["question"]:
        q_text = f"Question: {q}\nAnswer: "
        q_tokens = tokenizer(
            q_text,
            add_special_tokens=True,  # Include <s>
            truncation=False,
        )["input_ids"]
        # Subtract 1 because we don't include </s> in the prefix
        question_prefix_lengths.append(len(q_tokens) - 1)  # Exclude </s>
    
    result["question_prefix_length"] = question_prefix_lengths
    
    return result


# Tokenize datasets
train_dataset = Dataset.from_pandas(train_split_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_split_df, preserve_index=False)
test_dataset = Dataset.from_pandas(fake_test_df, preserve_index=False)

train_tokenized = train_dataset.map(
    tokenize_with_question_per_chunk,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_tokenized = val_dataset.map(
    tokenize_with_question_per_chunk,
    batched=True,
    remove_columns=val_dataset.column_names
)

test_tokenized = test_dataset.map(
    lambda x: tokenize_with_question_per_chunk(x, has_labels=True),
    batched=True,
    remove_columns=test_dataset.column_names
)

print(f"Train tokenized: {len(train_tokenized)} samples")
print(f"Val tokenized: {len(val_tokenized)} samples")
print(f"Test tokenized: {len(test_tokenized)} samples")

In [ ]:
@dataclass
class HierarchicalMultiHeadDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        clarity_labels = torch.tensor(
            [f.pop("clarity_labels") for f in features], 
            dtype=torch.long
        )
        evasion_labels = torch.tensor(
            [f.pop("evasion_labels") for f in features], 
            dtype=torch.long
        )
        question_prefix_lengths = torch.tensor(
            [f.pop("question_prefix_length") for f in features],
            dtype=torch.long
        )
        
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt"
        )
        batch["clarity_labels"] = clarity_labels
        batch["evasion_labels"] = evasion_labels
        batch["question_prefix_length"] = question_prefix_lengths
        
        return batch

data_collator = HierarchicalMultiHeadDataCollator(tokenizer=tokenizer)

In [ ]:
@dataclass
class MultiHeadClassifierOutput:
    loss: Optional[torch.FloatTensor] = None
    logits_clarity: torch.FloatTensor = None
    logits_evasion: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None
    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None


class MultiHeadHierarchicalMaxPoolingWithQuestionPerChunk(RobertaPreTrainedModel):
    
    @classmethod
    def _can_set_experts_implementation(cls) -> bool:
        """Override to prevent transformers from trying to inspect notebook source."""
        return False
    
    def __init__(self, config):
        super().__init__(config)
        self.num_clarity_labels = 3
        self.num_evasion_labels = 9
        self.config = config

        self.roberta = RobertaModel(config)

        classifier_dropout = (
            config.classifier_dropout 
            if hasattr(config, 'classifier_dropout') and config.classifier_dropout is not None 
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier_clarity = nn.Linear(config.hidden_size, self.num_clarity_labels)
        self.classifier_evasion = nn.Linear(config.hidden_size, self.num_evasion_labels)

        self.post_init()
    
    def create_chunks_with_question_prefix(
        self, 
        input_ids, 
        attention_mask, 
        question_prefix_length,
        max_length=512, 
        stride=256
    ):
        """
        Create chunks where each chunk contains the question prefix + a sliding window of the answer.
        
        Args:
            input_ids: [batch_size, seq_len] - full tokenized Q+A
            attention_mask: [batch_size, seq_len]
            question_prefix_length: [batch_size] - length of question prefix (including <s>)
            max_length: maximum chunk length (512)
            stride: stride for sliding window (256)
        """
        batch_size, seq_len = input_ids.shape
        
        all_chunk_ids = []
        all_chunk_masks = []
        chunk_to_sample = []
        
        for sample_idx in range(batch_size):
            sample_input_ids = input_ids[sample_idx]
            sample_attention_mask = attention_mask[sample_idx]
            q_prefix_len = question_prefix_length[sample_idx].item()
            
            # Find actual sequence length (excluding padding)
            actual_length = sample_attention_mask.sum().item()
            
            # Extract question prefix (including <s> and everything up to "Answer: ")
            # The prefix length includes <s> but not </s>
            question_prefix = sample_input_ids[:q_prefix_len]
            
            # Extract answer portion (everything after the question prefix)
            # Note: actual_length includes </s> at the end
            answer_start = q_prefix_len
            answer_tokens = sample_input_ids[answer_start:actual_length]
            
            # Calculate available space for answer in each chunk
            # max_length - q_prefix_len - 1 (for </s>)
            available_for_answer = max_length - q_prefix_len - 1
            
            if available_for_answer <= 0:
                # Question is too long, truncate and use single chunk
                chunk_ids = sample_input_ids[:max_length]
                chunk_mask = torch.ones(max_length, dtype=torch.long, device=input_ids.device)
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
                continue
            
            # Create chunks of answer
            answer_length = len(answer_tokens)
            
            if answer_length <= available_for_answer:
                # Answer fits in one chunk: <s> + q_prefix + answer + </s>
                chunk_ids = torch.cat([
                    question_prefix,
                    answer_tokens
                ])
                
                # Pad to max_length
                if len(chunk_ids) < max_length:
                    padding_length = max_length - len(chunk_ids)
                    chunk_ids = torch.cat([
                        chunk_ids,
                        torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                    ])
                    chunk_mask = torch.cat([
                        torch.ones(len(chunk_ids) - padding_length, dtype=torch.long, device=input_ids.device),
                        torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                    ])
                else:
                    chunk_mask = torch.ones(max_length, dtype=torch.long, device=input_ids.device)
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
            else:
                # Answer needs multiple chunks - use sliding window
                # Remove the final </s> from answer_tokens first
                answer_tokens_no_eos = answer_tokens[:-1] if answer_tokens[-1] == tokenizer.eos_token_id else answer_tokens
                
                start = 0
                while start < len(answer_tokens_no_eos):
                    end = min(start + available_for_answer, len(answer_tokens_no_eos))
                    answer_chunk = answer_tokens_no_eos[start:end]
                    
                    # Build chunk: question_prefix + answer_chunk + </s>
                    chunk_ids = torch.cat([
                        question_prefix,
                        answer_chunk,
                        torch.tensor([tokenizer.eos_token_id], dtype=torch.long, device=input_ids.device)
                    ])
                    
                    # Pad to max_length
                    if len(chunk_ids) < max_length:
                        padding_length = max_length - len(chunk_ids)
                        chunk_mask = torch.cat([
                            torch.ones(len(chunk_ids), dtype=torch.long, device=input_ids.device),
                            torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                        ])
                        chunk_ids = torch.cat([
                            chunk_ids,
                            torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                        ])
                    else:
                        chunk_mask = torch.ones(max_length, dtype=torch.long, device=input_ids.device)
                    
                    all_chunk_ids.append(chunk_ids)
                    all_chunk_masks.append(chunk_mask)
                    chunk_to_sample.append(sample_idx)
                    
                    start += stride
                    if end >= len(answer_tokens_no_eos):
                        break
        
        all_chunk_ids = torch.stack(all_chunk_ids, dim=0)
        all_chunk_masks = torch.stack(all_chunk_masks, dim=0)
        chunk_to_sample = torch.tensor(chunk_to_sample, dtype=torch.long, device=input_ids.device)
        
        return all_chunk_ids, all_chunk_masks, chunk_to_sample
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        question_prefix_length=None,
        clarity_labels=None,
        evasion_labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ) -> MultiHeadClassifierOutput:
        
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        
        batch_size = input_ids.shape[0]

        # Create chunks with question in each chunk
        all_chunk_ids, all_chunk_masks, chunk_to_sample = self.create_chunks_with_question_prefix(
            input_ids, attention_mask, question_prefix_length, max_length=512, stride=256
        )

        # Encode all chunks
        outputs = self.roberta(
            input_ids=all_chunk_ids,
            attention_mask=all_chunk_masks,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        # Extract <s> embeddings from each chunk
        all_cls_embeddings = outputs.last_hidden_state[:, 0, :]
        
        # Max pool across chunks for each sample
        batch_embeddings = []
        for sample_idx in range(batch_size):
            sample_mask = chunk_to_sample == sample_idx
            sample_chunk_embeddings = all_cls_embeddings[sample_mask]
            pooled_embedding = torch.max(sample_chunk_embeddings, dim=0)[0]
            batch_embeddings.append(pooled_embedding)

        pooled_output = torch.stack(batch_embeddings, dim=0)
        pooled_output = self.dropout(pooled_output)

        # Classification heads
        logits_clarity = self.classifier_clarity(pooled_output)
        logits_evasion = self.classifier_evasion(pooled_output)

        # Compute loss
        loss = None
        if clarity_labels is not None and evasion_labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss_clarity = loss_fct(logits_clarity.view(-1, self.num_clarity_labels), clarity_labels.view(-1))
            loss_evasion = loss_fct(logits_evasion.view(-1, self.num_evasion_labels), evasion_labels.view(-1))
            loss = loss_clarity + loss_evasion
        
        return MultiHeadClassifierOutput(
            loss=loss,
            logits_clarity=logits_clarity,
            logits_evasion=logits_evasion,
            hidden_states=None,
            attentions=None,
        )

In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(predictions, tuple):
        logits_clarity, logits_evasion = predictions
        if hasattr(logits_clarity, 'cpu'):
            logits_clarity = logits_clarity.cpu().numpy()
        if hasattr(logits_evasion, 'cpu'):
            logits_evasion = logits_evasion.cpu().numpy()
    else:
        logits_clarity = predictions[:, :3]
        logits_evasion = predictions[:, 3:]
        
    preds_clarity = np.argmax(logits_clarity, axis=-1)
    preds_evasion = np.argmax(logits_evasion, axis=-1)
    
    if isinstance(labels, tuple):
        labels_clarity, labels_evasion = labels
        if hasattr(labels_clarity, 'cpu'):
            labels_clarity = labels_clarity.cpu().numpy()
        if hasattr(labels_evasion, 'cpu'):
            labels_evasion = labels_evasion.cpu().numpy()
    else:
        labels_clarity = labels[:, 0] if labels.ndim > 1 else labels
        labels_evasion = labels[:, 1] if labels.ndim > 1 else labels

    valid_evasion_mask = labels_evasion != -100

    accuracy_clarity = accuracy_score(labels_clarity, preds_clarity)
    f1_clarity = f1_score(labels_clarity, preds_clarity, average='macro')

    if valid_evasion_mask.sum() > 0:
        accuracy_evasion = accuracy_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask])
        f1_evasion = f1_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask], average='macro')
    else:
        accuracy_evasion = 0.0
        f1_evasion = 0.0
    
    return {
        'accuracy_clarity': accuracy_clarity,
        'f1_clarity': f1_clarity,
        'accuracy_evasion': accuracy_evasion,
        'f1_evasion': f1_evasion,
        'f1_combined': (f1_clarity + f1_evasion) / 2,
    }

In [ ]:
class MultiHeadTrainer(Trainer):
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        clarity_labels = inputs.get('clarity_labels')
        evasion_labels = inputs.get('evasion_labels')
        
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.loss
            logits_clarity = outputs.logits_clarity
            logits_evasion = outputs.logits_evasion
        
        if prediction_loss_only:
            return (loss, None, None)
        
        logits = (logits_clarity.detach().float(), logits_evasion.detach().float())

        if clarity_labels is not None and evasion_labels is not None:
            labels = (clarity_labels.detach(), evasion_labels.detach())
        else:
            labels = None
        
        return (loss, logits, labels)


class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1_combined", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
# Initialize model
config = AutoConfig.from_pretrained(MODEL_NAME)
model = MultiHeadHierarchicalMaxPoolingWithQuestionPerChunk(config)

# Load pretrained weights
base_model = AutoModel.from_pretrained(MODEL_NAME)
model.roberta.load_state_dict(base_model.state_dict())
del base_model

print("Model initialized with pretrained RoBERTa-large weights")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results_multihead_maxpool_qperchunk",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    bf16=True,
    bf16_full_eval=True,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_combined",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

# Create trainer
trainer = MultiHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1_combined")],
)

print("Trainer created, starting training...")

In [ ]:
# Train the model
trainer.train()

In [ ]:
# Evaluate on validation set
val_results = trainer.evaluate()

print("\n" + "="*50)
print("VALIDATION RESULTS")
print("="*50)
print(f"Clarity - Accuracy: {val_results['eval_accuracy_clarity']:.4f}, F1: {val_results['eval_f1_clarity']:.4f}")
print(f"Evasion - Accuracy: {val_results['eval_accuracy_evasion']:.4f}, F1: {val_results['eval_f1_evasion']:.4f}")
print(f"Combined F1: {val_results['eval_f1_combined']:.4f}")
print("="*50)

In [ ]:
# Predict on fake test set
test_output = trainer.predict(test_tokenized)
logits_clarity_test, logits_evasion_test = test_output.predictions

# Get predicted classes
preds_clarity = np.argmax(logits_clarity_test, axis=-1)
preds_evasion = np.argmax(logits_evasion_test, axis=-1)

# Convert to labels
pred_clarity_labels = [clarity_id2label[p] for p in preds_clarity]
pred_evasion_labels = [evasion_id2label[p] for p in preds_evasion]

print(f"Generated predictions for {len(preds_clarity)} test samples")

In [ ]:
# Evaluate on fake test set (using the majority vote labels we created)
test_metrics = test_output.metrics

print("\n" + "="*50)
print("FAKE TEST SET RESULTS")
print("="*50)
print(f"Clarity - Accuracy: {test_metrics['test_accuracy_clarity']:.4f}, F1: {test_metrics['test_f1_clarity']:.4f}")
print(f"Evasion - Accuracy: {test_metrics['test_accuracy_evasion']:.4f}, F1: {test_metrics['test_f1_evasion']:.4f}")
print(f"Combined F1: {test_metrics['test_f1_combined']:.4f}")
print("="*50)

In [ ]:
# Create submission DataFrame
submission_df = fake_test_df[['question', 'interview_answer']].copy()
submission_df['pred_clarity'] = pred_clarity_labels
submission_df['pred_evasion'] = pred_evasion_labels
submission_df['true_clarity'] = fake_test_df['clarity_label']
submission_df['true_evasion'] = fake_test_df['evasion_label']

print("\nPrediction Distribution:")
print("\nClarity predictions:")
print(submission_df['pred_clarity'].value_counts())
print("\nEvasion predictions:")
print(submission_df['pred_evasion'].value_counts())

# Save predictions
submission_df.to_csv('predictions_fake_test_qperchunk.csv', index=False)
print("\nPredictions saved to predictions_fake_test_qperchunk.csv")

In [ ]:
# Detailed classification reports
from sklearn.metrics import classification_report

true_clarity = [clarity_label2id[l] for l in fake_test_df['clarity_label']]
true_evasion = [evasion_label2id[l] for l in fake_test_df['evasion_label']]

print("\n" + "="*50)
print("CLARITY CLASSIFICATION REPORT")
print("="*50)
print(classification_report(true_clarity, preds_clarity, target_names=clarity_labels))

print("\n" + "="*50)
print("EVASION CLASSIFICATION REPORT")
print("="*50)
print(classification_report(true_evasion, preds_evasion, target_names=evasion_classes))